# Website Summarizer and ChatBot using Ollama

## Overview

This is a simple application that extracts content from a website and generates a concise summary using Ollama. And also it contains a chatbot that explains the queries related to the website

## Features

- Extracts text content from the static website.
- Chatbot that answers the user queries related to the website

In [2]:
import requests
from bs4 import BeautifulSoup
import ollama
import pandas as pd

In [ ]:
url = "https://umeshchandra.in/blog"

response = requests.get(url)

soup = BeautifulSoup(response.text, "html.parser")

title = soup.title.text

paragraphs = [
    p.get_text(strip=True)
    for p in soup.find_all("p")
]

content = "\n".join(paragraphs)

print("Title:", title)
print(content[:500])

In [ ]:
MODEL="llama3.2"
response = ollama.chat(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": f"""
            Summarize the following webpage in 5 bullet points:

            {content}
            """
        }
    ]
)
print(response["message"]["content"])

## Website Summarizer

In [ ]:
question = input("Ask something about the website: ")

response = ollama.chat(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": f"""
            Website content:

            {content}

            Question:

            {question}
            """
        }
    ]
)

print(response["message"]["content"])

## Website Chatbot

In [17]:
def scrape_website(url):

    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")

    text = "\n".join(
        tag.get_text(strip=True)
        for tag in soup.find_all(["h1", "h2", "h3", "p", "li"])
    )

    return text

In [18]:
def ask_question(question):

    text = scrape_website(url)
    prompt = f"""
    You are an assistant.
    Answer the question using ONLY the website content below.
    If the answer is not present, say:
    "I could not find the answer on the website."
    Website content:
    {text}
    Question:
    {question}
    """

    response = ollama.chat(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response["message"]["content"]

In [ ]:
while True:

    question = input("\nAsk a question (type 'exit' to stop): ")

    if question.lower() == "exit":
        break

    answer = ask_question(question)

    print("\nBot:", answer)